# ЛР 4. ML-классификация стилей игроков

Шаги: 1) датасет → 2) признаки → 3) кластеризация → 4) churn → 5) важность признаков

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from lab04.make_dataset import make
from lab04.features import player_features
from lab04.models import cluster, churn_model

df = make()  # или pd.read_csv('../data/events.csv')
df.head()

## Шаг 2. Признаки

In [ ]:
F = player_features(df)
F.describe().T

## Шаг 3. Кластеризация (elbow / silhouette), интерпретация, 2D-визуализация (PCA/UMAP)

In [ ]:
k, scores, labels = cluster(F)
F['cluster'] = labels
print('best k =', k, scores)
F.groupby('cluster').mean().round(2)  # дайте кластерам имена

## Шаг 4. Churn-классификатор (ROC-AUC ≥ 0.70)

In [ ]:
y = df.groupby('player_id').churned.max().loc[F.index]
model, auc = churn_model(F.drop(columns='cluster'), y)
print('ROC-AUC =', round(auc, 3))

## Шаг 5. Важность признаков

In [ ]:
imp = pd.Series(model.feature_importances_, index=F.drop(columns='cluster').columns).sort_values()
imp.plot.barh(); plt.tight_layout(); plt.savefig('../results/feature_importance.png', dpi=150)